# P53 — Sobre las líneas y planos de ajuste más próximo a sistemas de puntos en el espacio

## 1. Título y paper

**Paper:** *On Lines and Planes of Closest Fit to Systems of Points in Space*  
**Autoría:** Karl Pearson  
**Año y venue:** 1901 · Philosophical Magazine, Series 6, 2(11), 559–572  
**Nivel:** L2 · **Motor:** `pca`  
**Ficha completa:** [`P53_pca`](../../papers/foundational/P53_pca/README.md)

**Hito:** La primera respuesta al problema de resumir una nube de puntos con menos dimensiones sin privilegiar ninguna variable.

- [DOI (Philosophical Magazine)](https://doi.org/10.1080/14786440109462720)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los mínimos cuadrados miden el error en vertical, y por tanto tratan una variable como causa y la otra como efecto. Cuando ninguna de las dos lo es, hay dos rectas distintas y ningún criterio para elegir.
2. Ejecutar una implementación mínima de la propuesta: Buscar la recta —o el plano— que minimiza la distancia perpendicular a los puntos. Esa dirección es simétrica en todas las variables y da los ejes principales.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Gauss y Legendre (1805–1809), mínimos cuadrados
- Galton (1886), regresión y correlación


## 4. Intuición

Una nube de puntos y la pregunta «¿cuál es la recta que mejor la resume?». Parece una sola pregunta y son tres, porque hay tres formas de medir la distancia de un punto a una recta: en vertical, en horizontal y por el camino más corto. Pearson toma la tercera.


## 5. Concepto mínimo

```text
Mínimos cuadrados de y sobre x : minimiza Σ (y − ŷ)²        ← error VERTICAL
Mínimos cuadrados de x sobre y : minimiza Σ (x − x̂)²        ← error HORIZONTAL
Eje principal (Pearson)        : minimiza Σ d⊥²             ← distancia PERPENDICULAR

    dirección del eje: la del mayor autovalor de la matriz de covarianzas
    tan(2θ) = 2·Sxy / (Sxx − Syy)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('pca', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Darán las tres rectas la misma pendiente?
2. ¿Cuál tendrá el menor error vertical?
3. ¿Cuál tendrá el menor error perpendicular?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('pca', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('pca', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Tres pendientes distintas —0,7782, 0,9097 y 1,0956— para los **mismos** diez puntos. Cada recta gana en su propio criterio y pierde en el ajeno: no hay una «mejor», hay una mejor **para cada error**. El eje de Pearson queda siempre entre las dos rectas de mínimos cuadrados.


## 10. Comentario pedagógico

Aquí nace la reducción de dimensionalidad. Si el primer eje explica el 92 % de la varianza, proyectar sobre él tira una dimensión y conserva casi toda la estructura. Eso es PCA, y es la misma idea que sostiene los espacios de representación de [P05](../../papers/foundational/P05_word2vec/README.md): direcciones que significan.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer la pendiente de mínimos cuadrados como «la relación» entre dos variables simétricas.


In [ ]:
r = run_paper_lab('pca', seed=7)['result']['rectas']
print('y sobre x :', r['minimos_cuadrados_y_sobre_x']['pendiente'])
print('x sobre y :', r['minimos_cuadrados_x_sobre_y']['pendiente'])
print('Las dos describen LA MISMA nube y no coinciden.')
print('Elegir una sin justificar el criterio es una decision oculta.')

## 12. Corrección

La versión que no privilegia ninguna variable, y el precio que paga:


In [ ]:
r = run_paper_lab('pca', seed=7)['result']
eje = r['rectas']['eje_principal_pearson']
ols = r['rectas']['minimos_cuadrados_y_sobre_x']
print('eje principal  -> perpendicular', eje['error_perpendicular'], '| vertical', eje['error_vertical'])
print('minimos cuadr. -> perpendicular', ols['error_perpendicular'], '| vertical', ols['error_vertical'])
print('Cada una gana en SU criterio. Eso no es un empate: es que la pregunta estaba incompleta.')

## 13. Desafío guiado

Comprueba con el resultado del motor que el eje principal queda entre las dos rectas de mínimos cuadrados en las 50 nubes perturbadas, y explica por qué eso tiene que pasar siempre.


In [ ]:
r = run_paper_lab('pca', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa PCA sobre un conjunto de datos real de más de dos dimensiones, proyecta a dos componentes y comprueba cuánta varianza conservas. Después escala las variables a media 0 y desviación 1 y repite: si el resultado cambia mucho, explica por qué.


## 15. Evidencia de aprendizaje

Guarda la tabla de las tres pendientes con sus dos errores y tu enunciado de qué criterio minimiza cada recta.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P53_pca/README.md) · evaluación formal: [`assessments/papers/P53_pca.md`](../../assessments/papers/P53_pca.md)


## 16. Cierre

Ya hay una forma de resumir datos con geometría. Falta una forma de calcular con ellos: la siguiente ficha reduce la neurona a una operación de umbral y demuestra que con eso basta para hacer lógica.


## 17. Conexión con el siguiente hito

- P05
- P43

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
